# Exercicis: diagnòstic de càncer de mama

**Optativa d'Aprenentatge automàtic — DAM/DAW 2n**

Fins ara heu mesurat els models amb un sol número: la precisió (percentatge
d'encerts). Aquest quadern fa servir un conjunt de dades mèdiques reals per
mirar-se aquest número de prop.

El conjunt és `load_breast_cancer` de scikit-learn: 569 tumors de mama, descrits
per 30 mesures fetes sobre la imatge de les seves cèl·lules (radi, textura,
perímetre, concavitat...), classificats com a **malignes** o **benignes**. És un
problema de classificació binària, com "spam o no spam".

Amb dues classes hi ha dues maneres d'equivocar-se, i no tenen per què valer el
mateix: pots dir "maligne" d'un tumor benigne, o dir "benigne" d'un de maligne.
Quina de les dues t'has de permetre menys aquí, i què hauries de mesurar per
tenir-la controlada, és la pregunta que tanca el quadern. No la responguis encara.

Fes els exercicis **en ordre i sense saltar-te'n cap ni llegir endavant**. Aquest
quadern està muntat perquè cada exercici et faci mirar l'anterior d'una altra
manera, i llegir-lo a salts li treu tota la gràcia. Tot funciona a Google Colab
sense instal·lar res.

## Les eines que necessites

Aquest quadern es fa amb aquestes eines. Cadascuna està explicada a
`EX_00_caixa_eines.ipynb`, amb la sintaxi i el parany que té. Aquí només tens la
llista: encaixar les peces és la feina de l'exercici.

Les que ja coneixes:
- `load_breast_cancer`, `train_test_split`, `StandardScaler`
- `KNeighborsClassifier`, `DecisionTreeClassifier`, `RandomForestClassifier`,
  `LogisticRegression`, `SVC`
- de pandas, `DataFrame` i `value_counts()`

**Les de mesurar no s'han vist a classe**, i les fas servir a partir de l'exercici
3. No les busquis als quaderns de teoria, que no hi són: estan totes explicades a
la caixa d'eines, amb un exemple mínim i el parany de cadascuna.
- `confusion_matrix` i `classification_report`
- `precision_score` i `recall_score`, i què vol dir `pos_label`
- `DummyClassifier`
- `predict_proba`, per a l'exercici 7

De numpy, `np.where` per construir prediccions a mà a l'exercici del llindar.

## Els objectes que manegues, i què els pots demanar

Les eines de la llista de dalt no treballen soltes: cadascuna et torna un
**objecte**, i el pas següent de l'exercici surt de demanar-li alguna cosa a aquell
objecte. Aquí tens, de cada un, el que faràs servir de veritat en aquest quadern:
els **atributs** (sense parèntesis) i els **mètodes** (amb parèntesis).

**El `Bunch` que torna `load_breast_cancer()`** — `sklearn.utils.Bunch`, una mena de
diccionari al qual també pots accedir amb punt (`data.target` és el mateix que
`data["target"]`):
- `.data` — les 569×30 mesures
- `.target` — les 569 etiquetes, en números (0 i 1)
- `.feature_names` — la llista amb els 30 noms de columna
- `.target_names` — els dos noms de diagnòstic, en el mateix ordre que els números;
  és l'única cosa que et diu quin número és `malignant` i quin `benign`
- `.frame` — només amb `as_frame=True`: el `DataFrame` amb les 30 columnes **i**
  `target`
- `.DESCR` — el text que descriu el dataset (`print(data.DESCR)` per llegir-lo)

**L'`ndarray` de NumPy** — el que hi ha dins de `X`, `y`, `X_entrena`, `X_examen`,
`prediccions_arbre`, `probabilitats`:
- `.shape` — la tupla (files, columnes); en un array d'una sola dimensió, `(n,)`
- `.dtype` — el tipus dels valors que hi ha a dins
- `.T` — la matriu transposada
- `.mean(axis=0)` i `.std(axis=0)` — mitjana i desviació **per columna**; amb
  `axis=1` per fila, i sense `axis` de tots els valors alhora
- `.argmax()` — la **posició** del valor més gran, no el valor
- `.reshape()` — canvia la forma sense tocar els valors
- `probabilitats[:, 0]` és la primera columna sencera: la probabilitat que el model
  dona a la classe `0`

**El `DataFrame` i la `Series` de pandas** — `dades`, `taula`:
- del `DataFrame`: `.shape`, `.columns`, `.dtypes` (atributs) i `.head()`,
  `.describe()`, `.sort_values(...)` (mètodes)
- de la `Series`: `.value_counts()`, `.unique()`
- `dades["target"]` et torna una columna com a `Series`

**Un model de scikit-learn** — `KNeighborsClassifier`, `DecisionTreeClassifier`,
`RandomForestClassifier`, `LogisticRegression`, `SVC` i també el
`DummyClassifier`. Tots tenen **la mateixa interfície**, i això és mitja
assignatura:
- `.fit(X, y)` — entrena amb aquestes dades; torna el model mateix
- `.predict(X)` — un array amb una etiqueta predita per fila
- `.predict_proba(X)` — una matriu de `(files, 2)` amb la probabilitat de cada
  classe; la columna `i` correspon a `.classes_[i]`. Els arbres, el bosc i la
  regressió logística en tenen; un `SVC` no, si no li has posat `probability=True`
- `.score(X, y)` — la precisió sobre aquestes dades, en un sol número
- `.classes_` — les classes que ha vist en entrenar, ordenades. És el que et diu
  quina columna de `predict_proba` és quina
- `.n_features_in_` — quantes columnes esperava; si li dones un nombre diferent a
  `.predict()`, peta
- `.feature_importances_` — **només** els models d'arbre (`DecisionTreeClassifier`,
  `RandomForestClassifier`): un pes per columna, i tots junts sumen 1
- `.coef_` i `.intercept_` — **només** `LogisticRegression` (i un `SVC` lineal)
- `.get_depth()` i `.get_n_leaves()` — només l'arbre

**L'`StandardScaler`** — no és un model, és un **transformador**, i per això té uns
altres mètodes:
- `.fit(X)`, `.transform(X)`, i els dos de cop amb `.fit_transform(X)`. No té
  `.predict()` ni `.score()`
- `.mean_` i `.scale_` — la mitjana i la desviació de cada columna que ha après de
  `X_entrena`. Són exactament les que fa servir després a `.transform(X_examen)`

**El que tornen les mesures** — no tot és un número:
- `confusion_matrix(...)` torna un **`ndarray`** de 2×2, i per tant té `.shape`,
  `.sum()` i indexació `matriu[fila, columna]`
- `classification_report(...)` torna un **text** per imprimir, no un objecte per
  remenar
- `precision_score`, `recall_score` i `accuracy_score` tornen un **float**

Dues regles que valen per a tot el quadern, perquè són les que fan petar el codi
sense que sembli que hi hagi res malament:

- **Un atribut va sense parèntesis i un mètode amb parèntesis.** `model.coef_` et
  dona els coeficients; `model.coef_()` dona `TypeError`, perquè no és una funció.
  I a l'inrevés: el `.describe` d'un DataFrame sense parèntesis no calcula res,
  t'ensenya el mètode.
- **El guió baix al final vol dir «això ho he après de les dades».** `.coef_`,
  `.feature_importances_` i `.classes_` **no existeixen abans de cridar `.fit()`**:
  si els demanes abans, salta un error. El mateix passa amb els atributs apresos
  de qualsevol transformador.

Per interrogar qualsevol d'aquests objectes pel teu compte —`type()`, `dir()`,
`objecte.metode?` i l'autocompletat amb TAB— tens
`FO_00_objectes_i_autocompletar.ipynb`.

### Exercici 1 — Carregar i explorar les dades

Carrega el dataset amb `load_breast_cancer(as_frame=True)`. `data.frame` ja
et dona un DataFrame amb les 30 columnes de mesures més una columna
`target`. Guarda'l a `dades` i mira quantes files i columnes té.

Després, compta quantes mostres hi ha de cada valor de `target`
(`dades["target"].value_counts()`). Fixa-t'hi: **el dataset no està
equilibrat**.

Finalment, el parany: `target` és un número (0 o 1), no diu per si sol si
és maligne o benigne. Mira `data.target_names` per esbrinar quin número és
quin diagnòstic, i escriu-ho en una frase.

In [ ]:
# Exercici 1
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

# 'dades' amb totes les columnes + target (data.frame ja el porta fet)
dades = None

# quantes files i columnes té 'dades'? (dades.shape)


# quantes mostres hi ha de cada valor de target?
# (dades["target"].value_counts())


# quins noms té cada valor de target? (data.target_names)

**Quin número és maligne i quin és benigne?**

_Escriu aquí la teva resposta, mirant `data.target_names`._

**Com saps que ho has fet bé:** 569 mostres i 31 columnes a `dades` (30 de
mesures més `target`). El repartiment és 357 benignes i 212 malignes: hi ha
un 62,7 % de benignes, més d'una i mitja vegada les malignes.

### Exercici 2 — Primer model, i la seva precisió

Separa les dades en entrenament i examen (`test_size=0.3, random_state=42`),
fent servir `data.data` com a `X` i `data.target` com a `y`. Entrena un
`DecisionTreeClassifier(random_state=42)` i mesura la seva precisió sobre
l'examen amb `.score()`.

Ara mira't el número que t'ha sortit i respon, **abans de continuar**: et sembla
bon model, aquest? El firmaries per fer-lo servir en un hospital? Contesta amb el
que et diu la precisió, que és l'única cosa que has mesurat.

Escriu-ho a la cel·la de sota i no la tornis a canviar: la rellegiràs al final del
quadern.

In [ ]:
# Exercici 2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = data.data
y = data.target

# 1. Parteix les dades amb train_test_split(X, y, test_size=0.3,
#    random_state=42)
X_entrena, X_examen, y_entrena, y_examen = None, None, None, None

# 2. Entrena model_arbre amb .fit(X_entrena, y_entrena) i mesura la
#    precisio sobre l'examen amb .score(X_examen, y_examen)
model_arbre = DecisionTreeClassifier(random_state=42)
precisio_arbre = None

print(f"Precisio de l'arbre a l'examen: {precisio_arbre}")

**Et sembla bon model, aquest? El firmaries?**

_Escriu aquí la teva resposta, dues línies. No la modifiquis més endavant._

**Com saps que ho has fet bé:** comprova la partició abans de creure't cap número.
`X_entrena` ha de tenir 398 files i `X_examen` 171, amb 30 columnes totes dues, i
398 + 171 ha de fer 569. Comprova també que has mesurat sobre l'examen i no sobre
l'entrenament: un `DecisionTreeClassifier` al qual no li limites la profunditat
encerta **sempre** el 100 % de l'entrenament, així que si et surt `1.0` estàs
mirant el conjunt equivocat. La precisió que t'hagi sortit a l'examen és la teva;
encara no la compares amb res.

### Exercici 3 — La matriu de confusió

La precisió és una divisió: encerts entre total. Pel camí perd **quins** errors ha
comès el model, i amb dues classes n'hi ha de dos tipus. La **matriu de confusió**
els ensenya tots: files = diagnòstic real, columnes = diagnòstic predit, en
l'ordre `[maligne, benigne]` (recorda l'exercici 1: `0 = maligne`).

Calcula-la amb `confusion_matrix(y_examen, prediccions, labels=[0, 1])` sobre el
model de l'exercici 2, i imprimeix també `classification_report`, que dona la
mateixa informació en forma de taula. Cap de les dues no s'ha vist a classe; les
tens a `EX_00_caixa_eines.ipynb`.

A la matriu, la casella `[fila=maligne, columna=benigne]` són els **falsos
negatius**: tumors que eren malignes i el model ha dit que eren benignes. El
pacient se'n va a casa sense tractament.

**Compta'ls tu a la matriu: quants n'hi ha?** Escriu el número a la cel·la de sota
i torna a llegir el que has respost a l'exercici 2.

In [ ]:
# Exercici 3
from sklearn.metrics import confusion_matrix, classification_report

# 1. Prediccions de model_arbre sobre X_examen (.predict(X_examen))
prediccions_arbre = None

# 2. confusion_matrix(y_examen, prediccions_arbre, labels=[0, 1]), i imprimeix-la

# 3. classification_report(y_examen, prediccions_arbre,
#    target_names=data.target_names), i imprimeix-lo

# 4. Agafa els falsos negatius de la casella que toca de la matriu:
#    la matriu es un ndarray, s'indexa matriu[fila, columna]
falsos_negatius_arbre = None

print(f"Falsos negatius: {falsos_negatius_arbre}")

**Quants falsos negatius té aquest model? I què en penses ara, del número de
l'exercici 2?**

_Escriu aquí el número i dues línies més._

**Com saps que ho has fet bé:** la matriu ha de ser 2×2 i les seves quatre
caselles han de sumar 171, el nombre de mostres de l'examen; si no sumen 171, has
comparat les prediccions d'un conjunt amb les etiquetes d'un altre. Segona
comprovació, que lliga la matriu amb l'exercici 2: la suma de la diagonal (els
encerts) dividida per 171 ha de donar **exactament** la precisió que et va sortir
abans. Si no coincideix, una de les dues coses està mal calculada.

Els falsos negatius, compta'ls tu de la matriu. El número no és aquí.

### Exercici 4 — El model ximple

Entrena un `DummyClassifier(strategy="most_frequent")`: un "model" que ni mira les
dades, sempre respon la classe més freqüent de l'entrenament. No s'ha vist a
classe; el tens a `EX_00_caixa_eines.ipynb`. Mesura'n la precisió sobre l'examen
igual que a l'exercici 2, i guarda'n també les prediccions, que les faràs servir
al següent.

**Serveix de res, aquest model?** Posa la seva precisió al costat de la de l'arbre
de l'exercici 2 i mira't la distància que hi ha entre les dues. Pensa també quants
falsos negatius deu tenir un model que sempre diu "benigne".

In [ ]:
# Exercici 4
from sklearn.dummy import DummyClassifier

model_ximple = DummyClassifier(strategy="most_frequent")

# 1. Entrena'l amb .fit(X_entrena, y_entrena) i calcula la precisio sobre
#    l'examen amb .score(X_examen, y_examen)
precisio_ximple = None

# 2. Guarda les seves prediccions sobre l'examen (.predict(X_examen)):
#    les fas servir a l'exercici 5
prediccions_ximple = None

print(f"Precisio del model ximple: {precisio_ximple}")

**Serveix de res aquest model?**

_Escriu aquí la teva resposta (dues o tres línies)._

**Com saps que ho has fet bé:** dues comprovacions, i cap número donat. Primera:
`prediccions_ximple` ha de valer el mateix a totes les 171 posicions
(`len(set(prediccions_ximple))` ha de ser `1`), i aquest valor ha de ser el de la
classe majoritària de l'entrenament; si no, no has entrenat el que et penses.
Segona: calcula-li la matriu de confusió amb `labels=[0, 1]` i mira-te'n la fila
dels malignes.

Després posa la seva precisió al costat de la de l'arbre i digues tu què vol dir la
distància que hi ha entre les dues.

### Exercici 5 — Dues mesures que la precisió no et dona

La precisió barreja els dos errors en un sol número. Aquestes dues els separen, i
totes dues es calculen **sobre una classe concreta**: aquí, la classe maligne, que
és l'etiqueta `0`, i per això hi va `pos_label=0`.

- `recall_score(y_examen, prediccions, pos_label=0)`, la **sensibilitat**: de tots
  els tumors que eren realment malignes, quina proporció n'ha trobat el model.
- `precision_score(y_examen, prediccions, pos_label=0)`, la **precisió del
  positiu**: de tots els que el model ha dit malignes, quina proporció ho era de
  veritat.

Mira't les definicions i esbrina, abans de calcular-les, quin dels dos tipus
d'error fa baixar cadascuna. Cap de les dues no s'ha vist a classe: les tens a
`EX_00_caixa_eines.ipynb`.

Calcula-les per a l'arbre de l'exercici 2 i també per al model ximple de
l'exercici 4.

I ara la pregunta que tanca la primera meitat del quadern: **en un diagnòstic de
càncer, quin dels dos errors no et pots permetre, i quina de les dues mesures
hauries de mirar en lloc de la precisió?** Respon-la amb els números que t'han
sortit, no de memòria.

In [ ]:
# Exercici 5
from sklearn.metrics import precision_score, recall_score

# 1. Sensibilitat i precisio del positiu de l'arbre (prediccions_arbre):
#    la sensibilitat es recall_score(...) i la precisio del positiu es
#    precision_score(...), totes dues amb pos_label=0
sensibilitat_arbre = None
precisio_positiu_arbre = None

# 2. Les mateixes dues mesures per al model ximple (prediccions_ximple)
sensibilitat_ximple = None
precisio_positiu_ximple = None

# 3. Comprova a ma que quadren amb les caselles de les matrius de confusio
#    que ja tens dels exercicis 3 i 4

print(f"Arbre  -> sensibilitat: {sensibilitat_arbre}, "
      f"precisio del positiu: {precisio_positiu_arbre}")
print(f"Ximple -> sensibilitat: {sensibilitat_ximple}, "
      f"precisio del positiu: {precisio_positiu_ximple}")

**Quin dels dos errors no et pots permetre en un diagnòstic de càncer, i quina
mesura hauries de mirar en lloc de la precisió?**

_Escriu aquí la teva resposta, tres o quatre línies, amb els números que t'han
sortit._

**Com saps que ho has fet bé:** que les dues mesures quadrin amb la matriu que ja
tens. Per a l'arbre, la sensibilitat sobre maligne ha de ser exactament la casella
`[maligne, maligne]` dividida per la suma de **tota la fila** dels malignes, i la
precisió del positiu, aquella mateixa casella dividida per la suma de **tota la
columna**. Fes les dues divisions a mà i comprova que et donen el mateix que
`recall_score` i `precision_score`; si et surten creuades, has canviat files per
columnes. Compara-ho també amb el `classification_report` de l'exercici 3, que ja
et donava aquests dos números a la fila `malignant` sense que ho sabessis.

Amb el model ximple hi trobaràs una cosa: no diu "maligne" ni una sola vegada, així
que la precisió del positiu no es pot calcular, perquè no hi ha cap positiu predit.
scikit-learn t'avisarà amb un `UndefinedMetricWarning` i et posarà un zero. No és
un error teu, i val la pena entendre d'on surt.

### Exercici 6 — Comparar els cinc models per sensibilitat, no per precisió

Entrena els cinc models de la teoria (k-NN, arbre de decisió, bosc aleatori,
regressió logística i SVM) sobre aquestes mateixes dades. Escala les dades amb
`StandardScaler` per a k-NN, la regressió logística i l'SVM (l'arbre i el bosc no
ho necessiten).

Per a cada model calcula tres números sobre l'examen: la **precisió**
(`accuracy_score`), la **sensibilitat sobre la classe maligne** i la **precisió del
positiu sobre la classe maligne**, aquestes dues amb `pos_label=0`, com a
l'exercici 5.

Munta una taula (un DataFrame amb una fila per model) i ordena-la primer per
precisió i després per sensibilitat. **Canvia el rànquing?**

In [ ]:
# Exercici 6
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score

# 1. Escala X_entrena i X_examen: .fit_transform() sobre l'entrenament i
#    .transform() sobre l'examen (l'escalador s'ajusta nomes amb
#    l'entrenament)
escalador = StandardScaler()
X_entrena_esc = None
X_examen_esc = None

# (model, necessita_escalar)
models = {
    "k-NN": (KNeighborsClassifier(n_neighbors=5), True),
    "Arbre de decisio": (DecisionTreeClassifier(random_state=42), False),
    "Bosc aleatori": (RandomForestClassifier(random_state=42), False),
    "Regressio logistica": (LogisticRegression(max_iter=10000, random_state=42), True),
    "SVM": (SVC(random_state=42), True),
}

# 2. Recorre models.items(): tria les X segons el booleà, entrena amb .fit(),
#    prediu amb .predict() i calcula accuracy_score(...),
#    recall_score(..., pos_label=0) i precision_score(..., pos_label=0).
#    Afegeix cada fila a 'files' amb .append()
files = []

taula = pd.DataFrame(
    files, columns=["model", "precisio", "sensibilitat_maligne", "precisio_maligne"]
)

# 3. Imprimeix la taula ordenada per precisio i despres ordenada per
#    sensibilitat_maligne (.sort_values("...", ascending=False) les dues
#    vegades), i compara els dos ranquings
taula

**Com saps que ho has fet bé:** la taula ha de tenir cinc files i tres números a
cada fila, cap d'ells buit. Comprova que l'escalador s'ha ajustat només amb
l'entrenament (`fit_transform` sobre `X_entrena` i després `transform` sobre
`X_examen`, mai `fit_transform` sobre l'examen), i que els dos models que no el
necessiten reben les dades sense escalar.

Dues coherències que han de sortir: l'arbre de decisió d'aquesta taula ha de donar
els **mateixos tres números** que ja tens dels exercicis 2 i 5 —mateixes dades,
mateixa llavor, cap escalat pel mig—, i cap dels cinc models no pot quedar per sota
del model ximple de l'exercici 4 en precisió. Si alguna de les dues falla, revisa el
bucle abans de mirar-te el rànquing.

Els dos rànquings, i si canvien o no, són la teva resposta.

### Exercici 7 — Moure el llindar de decisió

Fins ara, `predict()` decideix "maligne" o "benigne" triant la classe amb
més probabilitat, és a dir, tallant a 0,5. Però el 0,5 no és una llei
física: el pots moure.

Entrena una altra `LogisticRegression(max_iter=10000, random_state=42)`,
aquesta vegada **sense escalar** (per tenir un model amb marge de millora
i que l'efecte es vegi clar). Calcula la seva matriu de confusió amb el
llindar per defecte (`.predict()`).

Després, fes servir `predict_proba(X_examen)`: la primera columna és la
probabilitat que el model dona a "maligne" (comprova-ho amb
`model.classes_`, ha de ser `[0, 1]`). En lloc de predir amb `.predict()`,
construeix les teves prediccions dient "maligne" quan aquesta probabilitat
sigui **≥ 0,3** (enlloc de ≥ 0,5) i "benigne" en cas contrari. Calcula la
matriu de confusió d'aquestes noves prediccions.

**Compara les dues matrius: què passa amb els falsos negatius i amb els
falsos positius?**

In [ ]:
# Exercici 7
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

model_llindar = LogisticRegression(max_iter=10000, random_state=42)

# 1. Entrena'l sobre X_entrena / y_entrena, SENSE escalar (.fit())

# 2. Prediccions amb el llindar per defecte (.predict(X_examen)) i la seva
#    matriu de confusio (confusion_matrix(y_examen, ..., labels=[0, 1]))
prediccions_05 = None
matriu_05 = None

# 3. probabilitats = model_llindar.predict_proba(X_examen).
#    Mira model_llindar.classes_ per saber quina columna es "maligne"
probabilitats = None
llindar = 0.3

# 4. Prediu 0 (maligne) quan la probabilitat de maligne sigui >= llindar,
#    i 1 (benigne) en cas contrari (np.where ho fa en una linia)
prediccions_03 = None

# 5. La matriu de confusio de prediccions_03, amb els mateixos labels=[0, 1]
matriu_03 = None

print("Llindar per defecte (0,5):")
print(matriu_05)
print("Llindar 0,3:")
print(matriu_03)

**Com saps que ho has fet bé:** tres comprovacions del mecanisme, i cap matriu
donada. Primera: `model_llindar.classes_` ha de ser `[0, 1]`, que és el que et
garanteix que la columna 0 de `predict_proba` és la probabilitat de "maligne"; si
fos a l'inrevés, estaries movent el llindar de la classe equivocada. Segona, i és
la que val més: posa-hi `llindar = 0.5` i les teves `prediccions_03` han de sortir
**idèntiques** a `prediccions_05` (`np.array_equal(...)` ha de donar `True`),
perquè el llindar que fa servir `.predict()` és justament 0,5. Fes aquesta prova
abans de baixar-lo: és l'única manera de saber que has construït bé les prediccions
a mà. Tercera: les dues matrius han de sumar 171 caselles cadascuna.

Un cop tot això et quadra, torna a posar el 0,3, compara les dues matrius i digues
tu cap a on s'han mogut els falsos negatius i cap a on els falsos positius.

### Exercici 8 — La decisió final

Torna a llegir el que vas respondre a l'exercici 2, quan l'única cosa que tenies
era la precisió.

Ara tria, d'entre tot el que has provat, un model i un llindar de decisió.
Justifica'l per escrit en tres o quatre línies: **quin error prefereixes cometre,
més falsos positius o més falsos negatius, i per què**, en el context d'un
diagnòstic mèdic. No hi ha una única resposta correcta — es jutja per l'argument,
no pel model triat.

_Escriu aquí la teva decisió (tres o quatre línies)._

**Com saps que ho has fet bé:** no hi ha un número a comprovar aquí. La
resposta és bona si fa servir els números reals dels exercicis anteriors
(falsos negatius i falsos positius de com a mínim dos llindars) i explica
el motiu de la tria, no només el resultat.